# Project: Medical Diagnosis with Ensemble Methods

Use Bagging, Boosting, Voting, and Stacking ensembles for diagnosis prediction.

In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier, VotingClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
print('Libraries loaded')

In [ ]:
np.random.seed(42)
n = 3000
df = pd.DataFrame({
    'age': np.random.normal(55, 15, n).astype(int),
    'bmi': np.random.normal(28, 6, n),
    'blood_pressure': np.random.normal(130, 20, n),
    'cholesterol': np.random.normal(200, 40, n),
    'glucose': np.random.normal(100, 25, n),
    'smoker': np.random.choice([0,1], n, p=[0.6,0.4]),
    'exercise_hours': np.random.exponential(3, n).clip(0, 20),
    'family_history': np.random.choice([0,1], n, p=[0.7,0.3]),
})
log_odds = -5 + 0.03*(df['age']-50) + 0.05*(df['bmi']-25) +\
           0.02*(df['blood_pressure']-120) + 0.01*(df['cholesterol']-180) +\
           0.03*(df['glucose']-90) + 0.8*df['smoker'] - 0.3*df['exercise_hours'] +\
           0.5*df['family_history']
df['diagnosis'] = (1/(1+np.exp(-(log_odds + np.random.normal(0,1,n)))) > 0.3).astype(int)
print(f'Diagnosis rate: {df["diagnosis"].mean():.2%}')
df.head()

In [ ]:
X, y = df.drop('diagnosis', axis=1), df['diagnosis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## Compare Individual vs Ensemble

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}
results = []
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred, y_proba = model.predict(X_test_s), model.predict_proba(X_test_s)[:,1]
    results.append({'Model': name,
                    'Accuracy': accuracy_score(y_test, y_pred),
                    'F1': f1_score(y_test, y_pred),
                    'ROC-AUC': roc_auc_score(y_test, y_proba),
                    'CV (5-fold)': cross_val_score(model, X_train_s, y_train, cv=5, scoring='roc_auc').mean()})
print(pd.DataFrame(results).round(4).to_string(index=False))

## Voting and Stacking Ensembles

In [ ]:
voting = VotingClassifier(
    estimators=[('rf',RandomForestClassifier(n_estimators=100,random_state=42)),
                ('gb',GradientBoostingClassifier(n_estimators=100,random_state=42)),
                ('lr',LogisticRegression(max_iter=1000,random_state=42))],
    voting='soft')
stacking = StackingClassifier(
    estimators=[('rf',RandomForestClassifier(n_estimators=100,random_state=42)),
                ('gb',GradientBoostingClassifier(n_estimators=100,random_state=42)),
                ('svc',SVC(probability=True,random_state=42))],
    final_estimator=LogisticRegression(), cv=5)
for name, model in [('Voting',voting),('Stacking',stacking)]:
    model.fit(X_train_s, y_train)
    y_proba = model.predict_proba(X_test_s)[:,1]
    print(f'{name}: Accuracy={accuracy_score(y_test, model.predict(X_test_s)):.4f}, '
          f'ROC-AUC={roc_auc_score(y_test, y_proba):.4f}')

## Summary

- Compared individual vs ensemble models
- Tested Bagging (RF), Boosting (AdaBoost, GB)
- Voting and Stacking ensembles for best performance
- Cross-validation for robust evaluation